# HEST: query, edit IDs, and save clean patch data
All functions live in hest_download_pipeline.py. Run in Google Colab. Download only the patch H5 and expression H5AD for each selected slide into temporary local storage, match by barcode, and save PNG + CSV under slide/patch folders. No GPU required.

Images must be uint8 RGB, 224 x 224 x 3. Gene values are copied from AnnData.X without normalization, filtering, or log transformation; they are not guaranteed to be raw counts. Barcodes identify spots. Coordinate fields are included only when supplied by the source, without conversion.

Accept dataset access at https://huggingface.co/datasets/MahmoodLab/hest before logging in.

In [1]:
%pip install -q "huggingface_hub>=1.0,<2" "pandas>=2.2,<3" anndata h5py pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.


## 1. Load the functions and authenticate

In [1]:
from google.colab import drive
from pathlib import Path
import sys
import importlib

# Mount Drive before importing the companion .py module.
drive.mount("/content/drive")
FUNCTIONS_DIR = "/content/drive/MyDrive/HEST-1k"  # Change this folder.
assert (Path(FUNCTIONS_DIR) / "hest_download_pipeline.py").is_file(), "Check FUNCTIONS_DIR"
sys.path.insert(0, FUNCTIONS_DIR)
import hest_download_pipeline
importlib.reload(hest_download_pipeline)
from hest_download_pipeline import prepare_selection, download_clean_selection, clean_selection

from huggingface_hub import login
login("")

Mounted at /content/drive


## 2. Set your destination and preferences

In [7]:
OUTPUT_DIR = "/content/drive/MyDrive/HEST_clean"  # Only clean exports are saved here.
ORGANS = ["Breast"]
SPECIES = ["Homo sapiens"]
ST_TECHNOLOGIES = ["Visium"]
METADATA_FILENAME = "HEST_v1_3_0.csv"
MAX_SAMPLES = None
MAX_WORKERS = 2

## 3. Query sample IDs
This returns a plain Python list using your filters. No sample data is downloaded.

In [21]:
data_ids = prepare_selection(
    organs="Breast",
    species="Homo sapiens",
    st_technologies="Visium HD",
    metadata_filename=METADATA_FILENAME,
    max_samples=100,
)
print(data_ids)

Available organ: ['Bladder', 'Bone', 'Bowel', 'Brain', 'Breast', 'Cervix', 'Embryo', 'Eye', 'Heart', 'Kidney', 'Kidney/Brain', 'Liver', 'Lung', 'Lung/Brain', 'Lymph node', 'Lymphoid', 'Muscle', 'Other', 'Ovary', 'Pancreas', 'Placenta', 'Prostate', 'Skin', 'Spinal cord', 'Uterus', 'Whole organism']
Available species: ['Homo sapiens', 'Homo sapiens/Mus musculus', 'Mus musculus']
Available st_technology: ['Spatial Transcriptomics', 'Visium', 'Visium HD', "Visium HD 3'", 'Xenium', 'Xenium 5k']
Selected 3 samples
['TENX180', 'TENX162', 'TENX161']


## 4. Edit the list (optional)
Replace the list, remove IDs, or keep only a subset. The downloader uses this final list.

In [22]:
# Uncomment and edit to replace the query result:
# data_ids = ['TENX94', 'TENX99']
# Or keep only the first three:
# data_ids = data_ids[:3]
# Or remove an ID:
# data_ids = [i for i in data_ids if i != 'TENX94']
print('Final IDs:', data_ids)

Final IDs: ['TENX180', 'TENX162', 'TENX161']


## 5. Download and save clean data
For each slide, download patches + expression to local temporary storage, export by barcode, then discard the temporary originals. Existing data elsewhere stays intact. CSV contains all genes as gene,expression; barcode.csv contains the barcode and available coordinates. Missing barcode matches cause an error, never a positional fallback. Each slide has a patch manifest.

The destination slide folder must be empty to prevent overwriting old exports. If a run fails, review any partially exported folder and choose a new destination before retrying. PNG/CSV export sizes are additional to temporary H5 space. Google Drive receives many small files, so large cohorts can take time.

In [26]:
slide_report = download_clean_selection(
    data_ids,
    output_dir=OUTPUT_DIR,
    max_workers=MAX_WORKERS,
    metadata_filename=METADATA_FILENAME,
    staging_dir="/content",
)
display(slide_report)

Selected 1 samples, 2 files, 0.50 GiB total. Destination: /content/hest_stage_qzlzmqzf
         count        sum
path                     
patches      1  363603080
st           1  176704367


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Verified 2 files in /content/hest_stage_qzlzmqzf


/content/drive/MyDrive/HEST-1k/hest_download_pipeline.py:189: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = anndata.read_h5ad(expression_path, backed='r')


...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Verified 2 files in /content/hest_stage_mk4nb8fr


/content/drive/MyDrive/HEST-1k/hest_download_pipeline.py:189: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = anndata.read_h5ad(expression_path, backed='r')


KeyboardInterrupt: 

## Embedding process


In [ ]:
# RAW_DIR = "/content/drive/MyDrive/HEST_data"
# slide_report = clean_selection(data_ids, raw_dir=RAW_DIR, output_dir=OUTPUT_DIR)
# display(slide_report)


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import importlib

FUNCTIONS_DIR = Path("/content/drive/MyDrive/HEST-1k")

# Check the exact filename.
print([p.name for p in FUNCTIONS_DIR.glob("*preprocessing*")])
assert (FUNCTIONS_DIR / "hest_preprocessing.py").is_file(), (
    "hest_preprocessing.py was not found. Check its filename and folder."
)

sys.path.insert(0, str(FUNCTIONS_DIR))
importlib.invalidate_caches()

import hest_preprocessing
importlib.reload(hest_preprocessing)

from hest_preprocessing import (
    random_split_data,
    export_embeddings,
    export_gene_tensor,
)

Mounted at /content/drive
['hest_preprocessing.py']


In [34]:
from hest_preprocessing import random_split_data

train_paths, test_paths = random_split_data(
    data_folder="/content/drive/MyDrive/HEST_clean",
    train_percentage=80,  # 80% training, 20% testing
    seed=42,             # None for a different random split each run
)

Training: 5472 patches; testing: 1368 patches.
Skipped 1 incomplete patches.
Shuffled lists saved in: /content/drive/MyDrive/HEST_clean/splits/split_5d3a5c1db610


In [16]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/HEST_clean/splits/split_5d3a5c1db610/train.csv", nrows=500)
df.head()
df.to_csv('/content/drive/MyDrive/HEST_clean/splits/split_5d3a5c1db610/trainT.csv')


In [17]:
embeddings = export_embeddings(
    "/content/drive/MyDrive/HEST_clean/splits/split_5d3a5c1db610/trainT.csv",
    output_path="/content/drive/MyDrive/train_embeddings.pt",
    batch_size=128,
)

Loading weights:   0%|          | 0/607 [00:00<?, ?it/s]

Saved [500, 4608] tensor: /content/drive/MyDrive/train_embeddings.pt


In [11]:
genes = export_gene_tensor(
    "/content/drive/MyDrive/HEST_clean/splits/split_5d3a5c1db610/trainT.csv",
    output_path="/content/drive/MyDrive/train_genes.pt",
    embedding_path="/content/drive/MyDrive/train_embeddings.pt",
)

Saved [100, 18085, 2] tensor: /content/drive/MyDrive/train_genes.pt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')